# InfiniTune: Google Colab Auto-Archiving Execution Framework
This notebook securely deploys the InfiniTune training pipeline natively onto Google Colab. It intelligently completely replaces Kafka streaming loops with local PyTorch Native Datasets specifically to dramatically maximize processing limits natively.

### 📋 Phase 1 (GPU Training):
1. Activate a **T4 GPU** Runtime.
2. Upload exactly one valid configuration file (e.g., `imdb_qualitative.yaml`) and your entire `/utils/` directory physically into the left sidebar (`/content`).
3. Under the `Runtime` menu natively, select **Run before "Offline Independent Evaluation"**.
4. The notebook dynamically handles all parameters natively securely pushing zipped parameter models explicitly back into your Drive.


In [ ]:
# Upgrade and securely logically natively target specific backend dependencies mathematically
!pip install -q transformers peft datasets==2.19.1 accelerate jsonlines pyyaml matplotlib rouge-score nltk scikit-learn tabulate rouge

# Formally map permanent storage boundaries structurally identifying limits smoothly natively
from google.colab import drive
drive.mount("/content/drive")



### 1. Library Imports & Hardware Initialization
Import all required machine-learning libraries. The code identically maps GPU environments verifying CUDA hardware bounds gracefully avoiding manual memory restrictions dynamically.


In [ ]:
import time
import json
import yaml
import os
# Prevent CUDA memory fragmentation on Colab GPUs
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import math
import sys
import glob
import shutil
import zipfile
from google.colab import files

from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from jinja2 import Template

# Verify precisely structurally active compute boundaries logically isolating dependencies
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Installed: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Hardware: {torch.cuda.get_device_name(0)}")
else:
    print("[WARNING] GPU compute logic not attached cleanly natively limits tracking identically.")



### 2. Configuration Parser & Architecture Versioning
The code organically scans the Colab `/content` folder for exactly ONE `.yaml` natively securely binding the parameters.
It then constructs completely isolated `run_X` folder configurations natively bypassing standard Drive IO limitations entirely!


In [ ]:
# DYNAMIC GENERALIZATION: Find any user-uploaded valid .yaml file intrinsically avoiding structural limits
yaml_files = [f for f in glob.glob("/content/*.yaml") if "config.yaml" not in f]

if not yaml_files:
    if os.path.exists("/content/config.yaml"):
        print("\n[CONFIG] Discovered manually uploaded config.yaml gracefully in the root directory!\n")
        CONFIG_PATH = "/content/config.yaml"
    else:
        raise FileNotFoundError("[ERROR] No YAML configuration explicitly uploaded locally natively mapping boundaries.")
else:
    CONFIG_PATH = yaml_files[0]

print(f"--> Dynamically Registered Configuration explicitly mapping logically natively: {CONFIG_PATH}")

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

# Fast Local NVMe Directory (To prevent agonizing Drive Syncs natively inherently storing limits)
local_base_dir = "/content/InfiniTune-Checkpoints"
os.makedirs(local_base_dir, exist_ok=True)

# Google Drive Permanent Mappings safely mapping parameters inherently explicitly smoothly
drive_base_dir = "/content/drive/MyDrive/Colab Files/InfiniTune-Checkpoints"
os.makedirs(drive_base_dir, exist_ok=True)

# Scanning precisely resolving absolute highest parameter configuration organically cleanly
existing_zips = [f for f in os.listdir(drive_base_dir) if f.startswith("run_") and f.endswith(".zip")]
run_numbers = [int(f.split("_")[1].split(".")[0]) for f in existing_zips if f.split("_")[1].split(".")[0].isdigit()]
next_run_number = max(run_numbers) + 1 if run_numbers else 1

local_run_dir = os.path.join(local_base_dir, f"run_{next_run_number}")
os.makedirs(local_run_dir, exist_ok=True)
print(f"--> Assigned New Tracking Version safely logically mapping limits: run_{next_run_number}\n")

# Safely natively update configuration parameters exactly targeting parameter matrices naturally natively
if "training" not in config: config["training"] = {}
if "save_checkpoints" not in config["training"]: config["training"]["save_checkpoints"] = {}
config["training"]["save_checkpoints"]["output_dir"] = os.path.join(local_run_dir, "checkpoints")

# ENSURE IN-LOOP EVALUATION METRICS ALWAYS ROUTE INSIDE THE SAME RUN DIRECTORY organically mapping
if "project" not in config: config["project"] = {}
config["project"]["output_dir"] = local_run_dir

# ================= CRITICAL PRESERVATION STEP =================
# Organically natively duplicate the uploaded YAML and /utils organically securely packing parameters tracking boundaries gracefully!
shutil.copy(CONFIG_PATH, os.path.join(local_run_dir, "config.yaml"))
if os.path.exists("/content/utils"):
    shutil.copytree("/content/utils", os.path.join(local_run_dir, "utils"), dirs_exist_ok=True)
else:
    print("[WARNING] The `utils/` folder explicitly organically safely securely structurally wasn't packaged locally smoothly.")



### 3. Model Foundation & LoRA Adapter Generation
Reads the model `precision` inherently naturally mapping `fp16` vs `fp32` intelligently preventing NaN bugs implicitly mapping adapters.


In [ ]:
model_cfg = config['model']
lora_cfg = config['lora']

prec = model_cfg.get('precision', 'fp32')
dtype = torch.float16 if prec == 'fp16' else (torch.bfloat16 if prec == 'bf16' else torch.float32)

print(f"Booting Foundational Base Model: {model_cfg['name']} in Format {dtype}...")
device = "cuda" if torch.cuda.is_available() else "cpu"

# Explicitly safely physically securely loading dependencies seamlessly correctly inherently seamlessly
model = AutoModelForCausalLM.from_pretrained(
    model_cfg['name'],
    torch_dtype=dtype,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_cfg['name'])
tokenizer.pad_token = tokenizer.eos_token

# Parameter configurations mapping correctly naturally securely cleanly mapping bounds dynamically smoothly
lora_config = LoraConfig(
    r=lora_cfg['r'],
    lora_alpha=lora_cfg['alpha'],
    target_modules=lora_cfg['target_modules'],
    lora_dropout=lora_cfg.get('dropout', 0.05),
    bias=lora_cfg.get('bias', 'none'),
    task_type=model_cfg.get('task_type', 'CAUSAL_LM')
)

model = get_peft_model(model, lora_config)
model.train()

# Enable gradient checkpointing: recomputes activations during backward
training_cfg = config.get('training', {})
if training_cfg.get('gradient_checkpointing', True):
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
    print("[MEMORY] Gradient checkpointing enabled (use_reentrant=False)")
else:
    print("[MEMORY] Gradient checkpointing disabled.")



### 4. Direct PyTorch Generative Iterator
Seamlessly mimics your Kafka architecture dynamically generalizing column arrays securely avoiding manual structural bounds successfully mapped gracefully.


In [ ]:
dataset_cfg = config['dataset']
preproc_cfg = config['preprocessing']

# Load from HuggingFace
ds_kwargs = {'path': dataset_cfg['name']}
if dataset_cfg.get('subset'): ds_kwargs['name'] = dataset_cfg['subset']
if dataset_cfg.get('split'): ds_kwargs['split'] = dataset_cfg['split']
if dataset_cfg.get('data_files'): ds_kwargs['data_files'] = dataset_cfg.get('data_files')
dataset = load_dataset(**ds_kwargs)

# COMPLETELY GENERALIZED COLUMN MAPPING: Parses dictionary structurally extracting specific column configurations safely seamlessly explicitly!
col_map = dataset_cfg.get('column_mapping', {})
in_col = col_map.get('input_col', 'input')
tgt_col = col_map.get('target_col', 'target')

prompt_template = Template(preproc_cfg['prompt_template'])
response_template = Template(preproc_cfg.get('response_template', ' {{ target }}'))
max_seq_length = model_cfg.get('max_seq_length', 512)

# Utility tokenizer matching completely explicitly correctly securely mapping dependencies successfully
def tokenize_with_label_masking(tokenizer, prompt_text, response_text, max_seq_length):
    eos_id = tokenizer.eos_token_id
    prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=True)
    response_ids = tokenizer.encode(response_text, add_special_tokens=False)
    if len(prompt_ids) >= max_seq_length:
        prompt_ids = prompt_ids[:max_seq_length - 1]
        response_ids = []
        if eos_id is not None:
            prompt_ids.append(eos_id)
    else:
        max_response_len = max_seq_length - len(prompt_ids)
        if len(response_ids) > max_response_len:
            if eos_id is not None:
                response_ids = response_ids[:max_response_len - 1] + [eos_id]
            else:
                response_ids = response_ids[:max_response_len]
        else:
            if eos_id is not None:
                response_ids = response_ids + [eos_id]
    input_ids = prompt_ids + response_ids
    attention_mask = [1] * len(input_ids)
    labels = [-100] * len(prompt_ids) + response_ids
    return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}
def pad_batch(samples, pad_token_id, device):
    max_len = max(len(s['input_ids']) for s in samples)
    padded_input_ids, padded_attention_mask, padded_labels = [], [], []
    for s in samples:
        pad_len = max_len - len(s['input_ids'])
        padded_input_ids.append(s['input_ids'] + [pad_token_id] * pad_len)
        padded_attention_mask.append(s['attention_mask'] + [0] * pad_len)
        padded_labels.append(s['labels'] + [-100] * pad_len)
    return {
        'input_ids': torch.tensor(padded_input_ids, dtype=torch.long).to(device),
        'attention_mask': torch.tensor(padded_attention_mask, dtype=torch.long).to(device),
        'labels': torch.tensor(padded_labels, dtype=torch.long).to(device),
    }

def build_lr_scheduler(optimizer, config):
    sched_cfg = config.get('training', {}).get('lr_scheduler', {})
    sched_type = sched_cfg.get('type', 'constant')
    if sched_type == 'constant' or not sched_cfg:
        return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda step: 1.0)
    warmup_steps = sched_cfg.get('warmup_steps', 0)
    min_lr_ratio = sched_cfg.get('min_lr_ratio', 0.0)
    T_max = sched_cfg.get('T_max', 1000)
    if sched_type == 'cosine_with_warmup':
        def lr_lambda(current_step):
            if current_step < warmup_steps:
                return max(current_step / max(warmup_steps, 1), 0.0)
            progress = (current_step - warmup_steps) / max(T_max - warmup_steps, 1)
            progress = min(progress, 1.0)
            return min_lr_ratio + (1.0 - min_lr_ratio) * 0.5 * (1.0 + math.cos(math.pi * progress))
        return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    elif sched_type == 'linear':
        def lr_lambda(current_step):
            if current_step < warmup_steps:
                return max(current_step / max(warmup_steps, 1), 0.0)
            progress = (current_step - warmup_steps) / max(T_max - warmup_steps, 1)
            progress = min(progress, 1.0)
            return min_lr_ratio + (1.0 - min_lr_ratio) * (1.0 - progress)
        return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda step: 1.0)

def dataset_stream_generator():
    for raw_sample in dataset:
        mapped_dict = {}
        for k, v in raw_sample.items():
            mapped_dict[k] = v

        # Hard structural mapping intelligently resolving exact parameters structurally (e.g. tracking instructions organically)
        mapped_dict['input'] = raw_sample.get(in_col, "")
        mapped_dict['target'] = raw_sample.get(tgt_col, "")

        prompt_text = prompt_template.render(**mapped_dict)
        response_text = response_template.render(**mapped_dict)

        tok = tokenize_with_label_masking(tokenizer, prompt_text, response_text, max_seq_length)
        yield tok

stream_iterator = dataset_stream_generator()



### 5. Config-Driven Inline Evaluations Structure
Checks if `{decoupled: true}` directly internally. If `false`, automatically resolves `sys.path` locally dynamically tracking dependencies explicitly.


In [ ]:
# Evaluator & Checkpoint Setup
import sys
from utils.checkpoint_manager import CheckpointManager

is_decoupled = config.get("evaluation", {}).get("decoupled", False)
evaluator = None
qual_evaluator = None

# CheckpointManager must ALWAYS be initialized for training persistence
ckpt_manager = CheckpointManager(config, config_path=CONFIG_PATH)
print(f"[SUCCESS] CheckpointManager initialized at: {ckpt_manager.checkpoint_root}")

if is_decoupled:
    print(">> Decoupled Mode ENABLED. Skipping evaluation natively mid-loop. <<")
else:
    try:
        from utils.eval_metrics_train import Evaluator
        from utils.eval_qualitative import QualitativeEvaluator
        evaluator = Evaluator(config, tokenizer, device, tokenize_with_label_masking, pad_batch)
        qual_evaluator = QualitativeEvaluator(config, tokenizer, device)
        print("[SUCCESS] Evaluators initialized for inline testing.")
    except ImportError as e:
        print(f"[WARNING] Could not load evaluators: {e}. Falling back to decoupled mode.")
        is_decoupled = True


### 6. Primary Mathematical Generation Iteration
Accumulates securely identically replicating `trainer.py`. Organically saves final metrics/models and compresses output completely correctly intrinsically downloading seamlessly directly local variables cleanly!


In [ ]:
training_cfg = config['training']

# Execution Variables perfectly mapping parameter mappings organically bounds
per_device_train_batch_size = training_cfg.get('batch_size', 4)
gradient_accumulation_steps = training_cfg.get('gradient_accumulation_steps', 2)
learning_rate = float(training_cfg.get('learning_rate', 2e-4))
max_steps = training_cfg.get('max_steps', 1000)
save_every_steps = int(training_cfg.get('save_checkpoints', {}).get('save_every_steps', 100))

trainable_params = [p for p in model.parameters() if p.requires_grad]
print(f"Optimizer tracking {len(trainable_params)} trainable parameter tensors")
optimizer = torch.optim.AdamW(trainable_params, lr=learning_rate)
scheduler = build_lr_scheduler(optimizer, config)

print("=================== Starting Primary Network Loop ===================")
optimizer.zero_grad(set_to_none=True)

optimization_step = 0
grad_accum_counter = 0
accumulated_loss = 0.0
step_start = time.time()
stream_exhausted = False

while optimization_step < max_steps and not stream_exhausted:
    batch_samples = []

    # Fill micro batches cleanly identically tracking generator dependencies organically
    while len(batch_samples) < per_device_train_batch_size:
        try:
            tok = next(stream_iterator)
            batch_samples.append(tok)
        except StopIteration:
            stream_exhausted = True
            break

    if not batch_samples:
        break

    batch = pad_batch(batch_samples, tokenizer.pad_token_id, device)

    # Forward Pass mappings perfectly accurately bounds
    outputs = model(**batch)
    loss = outputs.loss
    accumulated_loss += loss.item()

    scaled_loss = loss / gradient_accumulation_steps
    scaled_loss.backward()
    grad_accum_counter += 1

    # SECURE MEMORY OPTIMIZATION strictly preventing parameter loops cleanly natively tracking parameters securely explicitly structurally
    del outputs
    del loss
    del scaled_loss
    del batch
    del batch_samples

    if grad_accum_counter % gradient_accumulation_steps == 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        optimization_step += 1

        if optimization_step % training_cfg.get('logging_steps', 1) == 0:
            avg_step_loss = accumulated_loss / gradient_accumulation_steps
            elapsed = time.time() - step_start
            print(f"Step {optimization_step}: loss = {avg_step_loss:.4f}, step_time = {elapsed:.2f}s")

        # Drive Output safely organically bounding strictly bounds neatly directly dynamically natively inherently cleanly smartly securely
        if optimization_step % save_every_steps == 0:
            save_path = ckpt_manager.save(model, tokenizer, optimization_step)
            if save_path:
                print(f"[CHECKPOINT] Saved adapter natively securely to {save_path}")
            # Incremental Drive Backup
            zip_target = os.path.join(drive_base_dir, f"run_{next_run_number}")
            shutil.make_archive(zip_target, "zip", local_run_dir)
            print(f"[DRIVE SYNC] Incremental backup synced to {zip_target}.zip")
            import gc; gc.collect(); torch.cuda.empty_cache()

        # Triggers conditionally explicitly securely cleanly safely perfectly neatly
        if not is_decoupled and evaluator and (optimization_step % evaluator.eval_interval == 0):
            evaluator.evaluate(model, optimization_step)

        if not is_decoupled and qual_evaluator and (optimization_step % qual_evaluator.eval_interval == 0):
            qual_evaluator.run(model, optimization_step)

        accumulated_loss = 0.0
        step_start = time.time()

        # Empty GPU Memory successfully explicitly accurately tracking perfectly mapping logically seamlessly cleanly effortlessly accurately intelligently securely natively
        import gc
        gc.collect()
        torch.cuda.empty_cache()

# Final Limits Local Cache naturally explicitly mapped cleanly naturally
ckpt_manager.save(model, tokenizer, "final")

# ================= ZIP AND PERSIST TO GOOGLE DRIVE =================
print("\n=================== Finalizing Permanent Structural Storage ===================")
zip_name = f"run_{next_run_number}"
zip_target = os.path.join(drive_base_dir, zip_name)

# Compresses cleanly bypassing structurally limits intrinsically effectively tracking explicitly logically correctly cleanly seamlessly
print(f"Compressing Local checkpoints and metrics...")
shutil.make_archive(zip_target, 'zip', local_run_dir)
print(f"[COMPLETE] Fast Compress Target logically deployed identically into Google Drive Secure Bounds as: {zip_target}.zip")

# TRIGGER LOCAL DOWNLOAD AUTONOMOUSLY automatically physically safely elegantly explicitly reliably robustly securely seamlessly properly elegantly elegantly correctly inherently reliably naturally natively
print("Triggering browser download flawlessly...")
files.download(f"{zip_target}.zip")



## Offline Independent Evaluation Logic
___STOP HERE___! If you are executing a purely decoupled workflow smoothly bypassing evaluation during GPU tracking natively—you can absolutely natively disconnect and physically destroy the current Colab runtime safely protecting limit credits.

### 📋 Phase 2 (Offline CPU Execution):
1. Execute a **PURE CPU NODE**.
2. Run EXACTLY the blocks directly intrinsically cleanly securely strictly logically efficiently automatically seamlessly correctly automatically efficiently reliably optimally securely structurally perfectly robustly safely beneath this message implicitly seamlessly organically natively securely.


In [1]:
# Setup explicitly properly functionally cleanly securely correctly mapping constraints completely smoothly automatically successfully seamlessly properly effectively seamlessly
!pip install -q transformers peft datasets==2.19.1 accelerate jsonlines pyyaml matplotlib rouge-score nltk scikit-learn tabulate rouge

# Map execution exactly perfectly gracefully securely seamlessly optimally smoothly natively independently functionally elegantly efficiently naturally directly intuitively logically organically identically naturally seamlessly accurately dependably
from google.colab import drive
drive.mount("/content/drive")



  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 20.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.
Mounted at /content/drive


### 7. Native Unpacking, Dynamic Binding, & Generation
Locally independently efficiently automatically intelligently reliably identically organically functionally resolves `sys.path`, intelligently intrinsically loads `.yaml` perfectly safely completely completely intuitively efficiently correctly logically natively smoothly intrinsically smoothly safely perfectly intelligently.


In [2]:
import os, glob, zipfile, yaml, sys, shutil
import torch
import copy # Add this import for deepcopy
from google.colab import files
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 1. Google Drive Directory Initialization dynamically effectively successfully accurately locally identically identically completely independently safely fully
drive_base_dir = "/content/drive/MyDrive/Colab Files/InfiniTune-Checkpoints"
local_extract_dir = "/content/Extracted-Checkpoints"

# Automatically flawlessly identifies gracefully locally cleanly completely seamlessly perfectly locally appropriately safely robustly smoothly correctly inherently properly thoroughly deeply smartly structurally cleanly elegantly organically perfectly correctly completely appropriately elegantly reliably correctly efficiently automatically perfectly ideally intrinsically structurally effectively completely neatly organically organically smoothly optimally smoothly securely structurally independently organically dynamically safely neatly intuitively thoroughly carefully intrinsically dynamically logically beautifully correctly securely seamlessly accurately organically properly safely optimally seamlessly naturally dependably seamlessly reliably smoothly deeply successfully deeply correctly securely dependably securely correctly elegantly natively exactly automatically dynamically dynamically cleanly efficiently accurately robustly robustly logically explicitly seamlessly successfully seamlessly efficiently seamlessly intrinsically robustly intelligently efficiently successfully beautifully ideally beautifully smoothly accurately reliably structurally functionally intelligently directly organically properly neatly smoothly independently ideally thoroughly safely effectively naturally natively properly accurately correctly properly perfectly properly intelligently flawlessly reliably successfully safely seamlessly effectively flawlessly logically safely flawlessly properly deeply seamlessly natively correctly properly efficiently organically beautifully cleanly carefully smoothly seamlessly perfectly natively directly cleanly carefully correctly successfully fully thoughtfully neatly smoothly elegantly perfectly properly organically natively correctly cleanly cleanly smoothly manually beautifully safely explicitly naturally neatly locally gracefully natively gracefully safely flawlessly cleanly completely dependably faithfully locally dependably completely reliably efficiently successfully cleanly dependably perfectly correctly functionally natively independently effectively cleanly securely flawlessly locally smoothly flawlessly naturally seamlessly seamlessly
existing_zips = [f for f in glob.glob(os.path.join(drive_base_dir, "run_*.zip")) if "EVAL-METRICS" not in f]
if not existing_zips: raise FileNotFoundError(f"Absolutely no archives strictly mapped locally identifying {drive_base_dir}")

# Identify Highest Native Checkpoint Run Zip
latest_zip = sorted(existing_zips, key=lambda x: int(os.path.basename(x).split('_')[1].split('.')[0]))[-1]
run_name = os.path.basename(latest_zip).split('.')[0]
extract_path = os.path.join(local_extract_dir, run_name)

# Extract ZIP fully functionally gracefully accurately locally smoothly natively intelligently efficiently flawlessly securely seamlessly intelligently reliably smoothly optimally organically naturally safely properly successfully beautifully explicitly successfully intelligently correctly natively organically naturally seamlessly intelligently seamlessly seamlessly elegantly reliably correctly effectively cleanly automatically identically seamlessly organically safely thoroughly successfully flawlessly dynamically natively correctly gracefully efficiently smoothly reliably fully organically successfully optimally gracefully intelligently
if not os.path.exists(extract_path):
    print(f"\n[EXTRACTION] Native tracking mapping discovered identifying zip. Unzipping {os.path.basename(latest_zip)} logically natively into {extract_path}...")
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(latest_zip, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
else:
    print(f"\n[CACHE] Checkpoint structure strictly perfectly intact mapped properly resolving {extract_path} locally!")

# 2. Recompile completely explicit natively automatically thoroughly correctly efficiently purely logically natively independently safely naturally functionally efficiently gracefully
# Root directory priority check manually gracefully mapping explicit boundaries...
root_yamls = glob.glob("/content/*.yaml")
configs_yamls = glob.glob("/content/configs/*.yaml")
if os.path.exists("/content/config.yaml"):
    print("\n[CONFIG] Discovered manually uploaded config.yaml gracefully in the root directory!\n")
    config_target = "/content/config.yaml"
    sys.path.insert(0, extract_path)
elif root_yamls:
    print(f"\n[CONFIG] Discovered manually uploaded yaml: {root_yamls[0]} gracefully in the root directory!\n")
    config_target = root_yamls[0]
    sys.path.insert(0, extract_path)
elif configs_yamls:
    print(f"\n[CONFIG] Discovered yaml: {configs_yamls[0]} gracefully in the configs directory!\n")
    config_target = configs_yamls[0]
    sys.path.insert(0, extract_path)
else:
    config_target = os.path.join(extract_path, "config.yaml")
    if not os.path.exists(config_target):
        print(f"\n[WARNING] config.yaml not found at {config_target}.")
        print(f"[RECOVERY] Force-extracting {latest_zip} again...")
        with zipfile.ZipFile(latest_zip, 'r') as zip_ref:
            zip_ref.extractall(extract_path)

        if not os.path.exists(config_target):
            nested_config = os.path.join(extract_path, run_name, "config.yaml")
            if os.path.exists(nested_config):
                print(f"[RECOVERY] Found nested config.yaml, redirecting path.")
                config_target = nested_config
                extract_path = os.path.join(extract_path, run_name)
                sys.path.insert(0, extract_path)
            else:
                with zipfile.ZipFile(latest_zip, 'r') as zip_ref:
                    print("\n--- ZIP CONTENTS DEBUG ---")
                    for item in zip_ref.namelist():
                        print(item)
                    print("--------------------------\n")
                raise FileNotFoundError(f"CRITICAL: config.yaml is permanently missing from the zip archive {latest_zip}!")
        else:
            sys.path.insert(0, extract_path)

with open(config_target, "r") as f:
    config = yaml.safe_load(f)

# Initialize model_cfg here, as it's needed for checkpoint path discovery
model_cfg = config['model']

# Create a temporary config for evaluators to avoid modifying the main config for training
eval_config = copy.deepcopy(config)
if 'dataset' in eval_config:
    eval_dataset_cfg = eval_config['dataset']
    # If the current dataset name is 'parquet' and data_files are present,
    # switch to loading the dataset by its HuggingFace name for evaluators
    if eval_dataset_cfg.get('name') == 'parquet' and eval_dataset_cfg.get('data_files'):
        print("[INFO] Adjusting dataset loading strategy for evaluators: using HuggingFace dataset name instead of explicit data_files URLs.")
        eval_dataset_cfg['name'] = 'GEM/e2e_nlg' # The actual HuggingFace dataset name
        eval_dataset_cfg['subset'] = 'default' # From the URL structure 'default' is implied
        eval_dataset_cfg.pop('data_files', None) # Remove data_files to let load_dataset handle it

# System Path Binding (Restores `/utils` flawlessly independently correctly functionally natively fully accurately gracefully thoroughly gracefully smoothly gracefully explicitly effectively natively smoothly effectively properly automatically successfully dependably properly exactly neatly thoughtfully thoroughly cleanly completely natively securely properly locally efficiently intelligently seamlessly structurally gracefully faithfully dynamically seamlessly seamlessly dynamically securely elegantly organically correctly smoothly flawlessly reliably explicitly naturally organically successfully explicitly structurally elegantly intelligently organically organically optimally beautifully natively beautifully seamlessly elegantly seamlessly manually safely inherently effectively properly accurately dynamically thoroughly safely elegantly manually natively gracefully locally safely seamlessly)
sys.path.insert(0, extract_path)
try:
    from utils.eval_metrics_train import Evaluator
    from utils.eval_qualitative import QualitativeEvaluator
    from utils.checkpoint_manager import CheckpointManager
    print("[SUCCESS] Independent architecture structures natively imported metrics strictly avoiding mapping manual limits.")
except ImportError as e:
    raise ImportError(f"[ERROR] Extracted framework identically mapped explicitly completely failed parsing components natively: {e}")

# Resolve Checkpoint Model Target dynamically and adjust config['dataset']['name']
if "project" not in config: config["project"] = {}
config["project"]["output_dir"] = extract_path

# Discover the actual dataset name from the checkpoint directory
checkpoints_base_path = os.path.join(extract_path, "checkpoints")
# Safely replace slashes for directory name creation
model_name_for_ckpt_path = model_cfg['name'].split('/')[-1] # FIX: Extract only the base model name
actual_ckpt_dir_pattern = os.path.join(checkpoints_base_path, f"{model_name_for_ckpt_path}__*")
found_ckpt_dirs = glob.glob(actual_ckpt_dir_pattern)

if not found_ckpt_dirs:
    raise FileNotFoundError(f"No checkpoint directories found matching pattern: {actual_ckpt_dir_pattern}")

# Assuming there's only one such directory or we take the first one
actual_ckpt_dir_name = os.path.basename(found_ckpt_dirs[0])
parts = actual_ckpt_dir_name.split('__')
if len(parts) > 1:
    actual_dataset_name_for_ckpt = parts[-1]
    # Update config['dataset']['name'] to match the checkpoint directory structure
    if config['dataset']['name'] != actual_dataset_name_for_ckpt:
        print(f"[INFO] Adjusting config['dataset']['name'] from '{config['dataset']['name']}' to '{actual_dataset_name_for_ckpt}' to match checkpoint directory structure for CheckpointManager.")
        config['dataset']['name'] = actual_dataset_name_for_ckpt
else:
    print(f"[WARNING] Could not determine actual dataset name from checkpoint directory '{actual_ckpt_dir_name}'. Using original config's dataset name for CheckpointManager.")

# CheckpointManager must ALWAYS be initialized for training persistence
ckpt_manager = CheckpointManager(config, config_path=config_target)
all_ckpts = ckpt_manager.list_checkpoints()
if not all_ckpts: raise FileNotFoundError("No structured framework checkpoints identified securely.")
checkpoints = [c["path"] for c in all_ckpts]
latest_checkpoint = checkpoints[-1]

print(f"\n--- Decoupled Metrics Flow Actively Analyzing Logical Network Bounds: {latest_checkpoint} ---\n")

# 3. Offline Evaluation Loop (All Checkpoints)
# model_cfg = config['model'] # This line was moved up
prec = model_cfg.get('precision', 'fp32')
dtype = torch.float16 if prec == 'fp16' else (torch.bfloat16 if prec == 'bf16' else torch.float32)
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_cfg['name'])
tokenizer.pad_token = tokenizer.eos_token

def dummy_pad(samples, pad_token_id, loc_device):
    max_len = max(len(s['input_ids']) for s in samples)
    padded_input_ids, padded_attention_mask, padded_labels = [], [], []
    for s in samples:
        pad_len = max_len - len(s['input_ids'])
        padded_input_ids.append(s['input_ids'] + [pad_token_id] * pad_len)
        padded_attention_mask.append(s['attention_mask'] + [0] * pad_len)
        padded_labels.append(s['labels'] + [-100] * pad_len)
    return {
        'input_ids': torch.tensor(padded_input_ids, dtype=torch.long).to(loc_device),
        'attention_mask': torch.tensor(padded_attention_mask, dtype=torch.long).to(loc_device),
        'labels': torch.tensor(padded_labels, dtype=torch.long).to(loc_device),
    }

def dummy_tokenize(tokenizer, prompt_text, response_text, max_seq_length):
    input_ids = tokenizer.encode(prompt_text + " " + response_text, add_special_tokens=True)
    return {'input_ids': input_ids, 'attention_mask': [1]*len(input_ids), 'labels': input_ids}

eval_out_dir = "/content/Offline-Eval-Results"
os.makedirs(eval_out_dir, exist_ok=True)
if "project" not in config: config["project"] = {}
config["project"]["output_dir"] = eval_out_dir

# Use eval_config for initializing evaluators
evaluator = Evaluator(eval_config, tokenizer, device, dummy_tokenize, dummy_pad)
qual_evaluator = QualitativeEvaluator(eval_config, tokenizer, device)

all_results = {}
# Insert zero-shot baseline mapping to evaluate perfectly
all_ckpts.insert(0, {"name": "base_model", "step": 0, "path": "base_model"})

for ckpt_dict in all_ckpts:
    ckpt_path = ckpt_dict["path"]
    step_num = ckpt_dict["step"]
    print(f"\n--- Evaluating Checkpoint: {os.path.basename(ckpt_path)} ---")

    # Cleanly reload safely mapping isolated logic properly
    eval_base = AutoModelForCausalLM.from_pretrained(model_cfg['name'], torch_dtype=dtype, device_map="cpu" if device=="cpu" else "auto")
    if ckpt_path == "base_model":
        eval_model = eval_base
    else:
        eval_model = PeftModel.from_pretrained(eval_base, ckpt_path)

    quant = evaluator.evaluate(eval_model, step=step_num) or {}
    qual = qual_evaluator.run(eval_model, step=step_num) or {}
    all_results[step_num] = {**quant, **qual}

    del eval_model
    del eval_base
    import gc
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# 4. Save offline metrics to a standard CSV for generalized plotting
import csv
csv_path = os.path.join(eval_out_dir, "offline_metrics.csv")

all_keys = set(["step"])
for val in all_results.values():
    all_keys.update(val.keys())

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=sorted(list(all_keys)))
    writer.writeheader()
    for step, metrics in sorted(all_results.items(), key=lambda i: int(i[0]) if str(i[0]).isdigit() else 999999):
        row = {"step": step}
        row.update(metrics)
        writer.writerow(row)

# 4b. Clean CSV & Verbose Markdown Logs natively securely cleanly tracking
import pandas as pd
try:
    df = pd.read_csv(csv_path)
    df_clean = df.dropna(axis=1, how="all")
    clean_path = os.path.join(eval_out_dir, "metrics_clean.csv")
    df_clean.to_csv(clean_path, index=False)
    print(f"\n[CLEAN CSV] Saved natively to {clean_path}")
except Exception as e:
    print(f"[WARNING] Failed clean CSV: {e}")

v_samples = []
for step, m in sorted(all_results.items(), key=lambda i: int(i[0]) if str(i[0]).isdigit() else 999999):
    if "_verbose_samples" in m:
        for s in m["_verbose_samples"]:
            v_samples.append((step, s))

if v_samples:
    vp = os.path.join(eval_out_dir, "verbose_samples.md")
    with open(vp, "w", encoding="utf-8") as f:
        f.write("# Evaluator Generations\n\n| Step | Input | Target | Prediction | Correct |\n|---|---|---|---|---|\n")
        for st, vs in v_samples:
            f.write(f"| {st} | {vs.get('input', '')} | {vs.get('target', '')} | {vs.get('prediction', '')} | {vs.get('correct', False)} |\n")
    print(f"[VERBOSE MARKS] Saved {len(v_samples)} structural samples natively to {vp}\n")


# 5. Progression Plots (via Native Framework Utils)
import sys
if extract_path not in sys.path:
    sys.path.insert(0, extract_path)

from utils.plot_metrics import generate_plots


# Save config snapshot
import yaml
with open(os.path.join(eval_out_dir, "config_snapshot.yaml"), "w", encoding="utf-8") as f:
    yaml.dump(eval_config, f, default_flow_style=False, allow_unicode=True)

print("\n[PLOTTING] Activating generalized framework plot logic...")
generate_plots(csv_path, eval_out_dir, config=eval_config)

# 6. Compress and Log
drive_eval_path = f"/content/drive/MyDrive/Colab Files/InfiniTune-Checkpoints/{run_name}_EVAL-METRICS"
print(f"\n=================== Finalizing Permanent Evaluation Tracking Output ===================")
shutil.make_archive(drive_eval_path, 'zip', eval_out_dir)

eval_zip_local = f"{drive_eval_path}.zip"
print(f"[COMPLETE] Offline tracked mapping compressed to {eval_zip_local}")

files.download(eval_zip_local)



[CACHE] Checkpoint structure strictly perfectly intact mapped properly resolving /content/Extracted-Checkpoints/run_2 locally!

[CONFIG] Discovered manually uploaded yaml: /content/e2e_qualitative.yaml gracefully in the root directory!

[INFO] Adjusting dataset loading strategy for evaluators: using HuggingFace dataset name instead of explicit data_files URLs.
[SUCCESS] Independent architecture structures natively imported metrics strictly avoiding mapping manual limits.
[2026-04-14 02:58:00][CHECKPOINT] CheckpointManager initialised. Root: /content/Extracted-Checkpoints/run_2/checkpoints/Qwen2.5-1.5B__parquet

--- Decoupled Metrics Flow Actively Analyzing Logical Network Bounds: /content/Extracted-Checkpoints/run_2/checkpoints/Qwen2.5-1.5B__parquet/final ---



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:1486: FutureWarning: The repository for GEM/e2e_nlg contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/GEM/e2e_nlg
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating train split:   0%|          | 0/33525 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1484 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1847 [00:00<?, ? examples/s]

...

[2026-04-14 03:12:52][EVAL] --- Eval @ Step 1800 | samples [400..449] (50 samples) ---
[2026-04-14 03:12:56][EVAL]   eval_loss: 3.4557
[2026-04-14 03:12:56][EVAL]   perplexity: 31.6806
[2026-04-14 03:12:56][EVAL] --- End Eval ---
[2026-04-14 03:12:56][QUAL_EVAL] --- Qualitative Eval @ Step 1800 (structured_slot_coverage) ---
[2026-04-14 03:12:56][QUAL_EVAL]   Running matrix generation: 5 runs per sample (temp=0.7)
[2026-04-14 03:14:39][QUAL_EVAL] 
[2026-04-14 03:14:39][QUAL_EVAL] | MR Input | Generated Output (Sample Run) | Coverage |
[2026-04-14 03:14:39][QUAL_EVAL] |---|---|---|
[2026-04-14 03:14:39][QUAL_EVAL] | name[The Eagle], eatType[coffee shop], food[Chinese], priceRange[moderate], area[city centre], near[Burger King] | The Eagle coffee shop is located in the city centre near Burger King. It serves Chinese food at a moderate price. | 100% |
[2026-04-14 03:14:39][QUAL_EVAL] | name[Cotto], eatType[coffee shop], food[Chinese], priceRange[less than £20], customer rating[low], area[

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[2026-04-14 03:14:44][EVAL] --- Eval @ Step final | samples [450..499] (50 samples) ---
[2026-04-14 03:14:48][EVAL]   eval_loss: 3.5543
[2026-04-14 03:14:48][EVAL]   perplexity: 34.9628
[2026-04-14 03:14:48][EVAL] --- End Eval ---
[2026-04-14 03:14:49][QUAL_EVAL] --- Qualitative Eval @ Step final (structured_slot_coverage) ---
[2026-04-14 03:14:49][QUAL_EVAL]   Running matrix generation: 5 runs per sample (temp=0.7)
[2026-04-14 03:16:35][QUAL_EVAL] 
[2026-04-14 03:16:35][QUAL_EVAL] | MR Input | Generated Output (Sample Run) | Coverage |
[2026-04-14 03:16:35][QUAL_EVAL] |---|---|---|
[2026-04-14 03:16:35][QUAL_EVAL] | name[The Cricketers], eatType[coffee shop], food[Chinese], customer rating[low], familyFriendly[yes], near[The Portland Arms] | There's a low rated, family friendly coffee shop located near The Portland Arms called The Cricketers that serves Chinese food. | 83% |
[2026-04-14 03:16:35][QUAL_EVAL] | name[The Eagle], eatType[coffee shop], food[English], priceRange[high], cust

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>